# Week 3 — Control Structures: Loops & Conditionals
### Python for Blockchain Analytics | Phase 1

---

**What you'll learn this week:**
- `if / elif / else` — making decisions based on on-chain data
- Comparison and logical operators
- `for` loops — processing lists of transactions or wallets
- `while` loops — polling until a condition is met
- `break`, `continue`, `range()`
- Nesting — loops inside loops, conditions inside loops

**SQL analyst parallel:**
- `if/elif/else` ≈ `CASE WHEN ... THEN ... ELSE ... END`
- `for` loop over a list ≈ a query returning multiple rows — you process each row
- `while` ≈ a retry loop around a query that might fail

#### Note: Always play around with values to understand how Python evaluates conditions.
---

## 1. if / elif / else

The most fundamental decision structure. Python evaluates a condition — if it's `True`, it runs that block; otherwise it moves on.

**Blockchain analogy:** A smart contract's `require()` statement is basically an `if` — if the condition isn't met, revert. In Python, if the condition isn't met, we take a different code path.

In [4]:
# Basic if statement
eth_price = 2456.9

if eth_price > 3000:
    print("ETH is above $3,000")

# Note: the indented block only runs if the condition is True
# Nothing happens if it's False — no error, just skipped
#change the value of eth_price to see the difference in output

In [7]:
# if / else — two paths
wallet_balance_eth = 0.05

if wallet_balance_eth >= 0.1:
    print("Sufficient balance to cover gas")
else:
    print("Insufficient balance — top up needed")

Insufficient balance — top up needed


In [12]:
# if / elif / else — multiple conditions
# SQL parallel: CASE WHEN price > 5000 THEN 'high' WHEN price > 2000 THEN 'mid' ELSE 'low' END

eth_price = 3247.85

if eth_price >= 5000:
    market_label = "Bull run"
elif eth_price >= 3000:
    market_label = "Strong"
elif eth_price >= 1500:
    market_label = "Mid range"
elif eth_price >= 500:
    market_label = "Bearish"
else:
    market_label = "Capitulation"

print(f"ETH at ${eth_price:,.2f} → {market_label}")

ETH at $3,247.85 → Strong


### Important: only one branch runs

Python checks conditions top to bottom and runs the **first** one that is `True`. Once a match is found, it skips all the rest.

This matters when conditions overlap — order them from most specific to least specific.

In [14]:
# Wallet classifier — classify by transaction count
# SQL parallel: CASE WHEN tx_count > 10000 THEN 'bot' ...

def classify_wallet(tx_count, balance_eth):
    """Classify a wallet based on activity and balance."""
    if tx_count > 10_000:
        return "🤖 Bot / High-frequency trader"
    elif balance_eth > 1000:
        return "🐋 Whale"
    elif balance_eth > 10:
        return "🐬 Dolphin"
    elif tx_count > 500:
        return "🦈 Active trader"
    elif tx_count > 50:
        return "👤 Regular user"
    else:
        return "🐣 New / Inactive"

# Test it
wallets = [
    ("0xd8dA...6045", 12_500, 5.2),
    ("0x1f98...984",  200,    1450.0),
    ("0xAbC1...001",  800,    3.4),
    ("0x0000...dead", 12,     0.001),
]

for address, tx_count, balance in wallets:
    label = classify_wallet(tx_count, balance)
    print(f"{address} | {tx_count:>7,} txns | {balance:>8.3f} ETH | {label}")

0xd8dA...6045 |  12,500 txns |    5.200 ETH | 🤖 Bot / High-frequency trader
0x1f98...984 |     200 txns | 1450.000 ETH | 🐋 Whale
0xAbC1...001 |     800 txns |    3.400 ETH | 🦈 Active trader
0x0000...dead |      12 txns |    0.001 ETH | 🐣 New / Inactive


## 2. Comparison and Logical Operators

### Comparison operators

| Operator | Meaning | Example |
|----------|---------|---------|
| `==` | Equal to | `status == "success"` |
| `!=` | Not equal | `token != "ETH"` |
| `>` | Greater than | `price > 3000` |
| `<` | Less than | `gas < 50` |
| `>=` | Greater than or equal | `balance >= min_balance` |
| `<=` | Less than or equal | `slippage <= 0.005` |

### Logical operators

| Operator | Meaning | Example |
|----------|---------|---------|
| `and` | Both must be True | `is_verified and has_liquidity` |
| `or` | At least one True | `is_whale or is_bot` |
| `not` | Flip True/False | `not is_flagged` |

In [23]:
# Logical operators in DeFi context
token_price   = 1.0002
liquidity_usd = 2_500_000
is_verified   = True
is_flagged    = False
daily_volume  = 850_000

# when 'and' is used — all conditions must pass (like a smart contract require() chain)
is_safe_to_trade = (
    is_verified
    and not is_flagged
    and liquidity_usd > 1_000_000
    and daily_volume > 100_000
)
print(f"Safe to trade: {is_safe_to_trade}")

# when or is used — any condition triggers the alert
needs_review = (
    liquidity_usd < 100_000
    or daily_volume < 10_000
    or is_flagged
)
print(f"Needs review: {needs_review}")

# Combining — mimics complex WHERE clauses
# SQL: WHERE is_verified = TRUE AND (liquidity > 1M OR volume > 500k)
good_token = is_verified and (liquidity_usd > 1_000_000 or daily_volume > 500_000)
print(f"Good token: {good_token}")

Safe to trade: True
Needs review: False
Good token: True


In [30]:
# in operator — This is used to check membership (very useful for blockchain)
STABLECOINS = ["USDC", "USDT", "DAI", "FRAX", "LUSD"]
MAJOR_TOKENS = ["ETH", "BTC", "BNB", "SOL"]

token = "BTC"

if token in STABLECOINS:
    print(f"{token} is a stablecoin — there is no price volatility risk")
elif token in MAJOR_TOKENS:
    print(f"{token} is a major token - there will be some price volatility risk")
else:
    print(f"{token} is an altcoin — There will be high volatility risk")

# using not in operator (combining with NOT with IN operator to check exclusion)
blacklisted = ["0xDead...001", "0xScam...999"]
wallet = "0xLegit...abc"

if wallet not in blacklisted:
    print("Wallet is clean — It is not blacklisted")

BTC is a major token - there will be some price volatility risk
Wallet is clean — It is not blacklisted


## 3. for Loops

A `for` loop repeats a block of code for each item in a sequence picking it one at a time and iterating till the end

**SQL parallel:** Think of a list as a query result with multiple rows. A `for` loop processes each row, one at a time — like applying logic row by row that you'd normally express with a CASE WHEN or a window function.

In [31]:
# Basic for loop — iterate over a list
tokens = ["ETH", "BTC", "USDC", "UNI", "AAVE"]

for token in tokens:
    print(f"Processing: {token}")

Processing: ETH
Processing: BTC
Processing: USDC
Processing: UNI
Processing: AAVE


In [32]:
# Loop with index using enumerate()
# enumerate() gives you (position, value) pairs
transactions = [
    "0xabc...001",
    "0xdef...002",
    "0xghi...003",
]

for i, tx_hash in enumerate(transactions):
    print(f"Transaction #{i + 1}: {tx_hash}")

Transaction #1: 0xabc...001
Transaction #2: 0xdef...002
Transaction #3: 0xghi...003


In [33]:
# Looping over a list of dicts — like iterating query result rows
# SQL parallel: each dict is one row from SELECT * FROM dex_trades

trades = [
    {"token": "ETH",  "amount": 2.5,    "price_usd": 3247.85, "side": "buy"},
    {"token": "UNI",  "amount": 500,    "price_usd": 12.84,   "side": "sell"},
    {"token": "USDC", "amount": 10_000, "price_usd": 1.00,    "side": "buy"},
    {"token": "AAVE", "amount": 15,     "price_usd": 98.50,   "side": "sell"},
    {"token": "ETH",  "amount": 1.0,    "price_usd": 3251.20, "side": "sell"},
]

total_buy_volume  = 0
total_sell_volume = 0

for trade in trades:
    value_usd = trade["amount"] * trade["price_usd"]
    side_icon = "🟢" if trade["side"] == "buy" else "🔴"

    print(f"{side_icon} {trade['side'].upper():4} | {trade['token']:4} | "
          f"{trade['amount']:>10,.2f} | ${value_usd:>12,.2f}")

    if trade["side"] == "buy":
        total_buy_volume += value_usd
    else:
        total_sell_volume += value_usd

print("-" * 52)
print(f"{'Total Buy Volume:':30} ${total_buy_volume:>12,.2f}")
print(f"{'Total Sell Volume:':30} ${total_sell_volume:>12,.2f}")
net = total_buy_volume - total_sell_volume
direction = "Buy pressure 🟢" if net > 0 else "Sell pressure 🔴"
print(f"{'Net:':30} ${net:>12,.2f} — {direction}")

🟢 BUY  | ETH  |       2.50 | $    8,119.62
🔴 SELL | UNI  |     500.00 | $    6,420.00
🟢 BUY  | USDC |  10,000.00 | $   10,000.00
🔴 SELL | AAVE |      15.00 | $    1,477.50
🔴 SELL | ETH  |       1.00 | $    3,251.20
----------------------------------------------------
Total Buy Volume:              $   18,119.62
Total Sell Volume:             $   11,148.70
Net:                           $    6,970.92 — Buy pressure 🟢


## 4. range()

`range()` generates a sequence of numbers. Perfect for when you need to loop a specific number of times — like processing blocks in a range.

In [38]:
# range(stop) — 0 up to (not including) stop
for i in range(5):
    print(i, end=" ")   # end=" " keeps output on one line
print()  # newline

# range(start, stop)
for block in range(19_847_290, 19_847_296):
    print(f"Processing block {block:,}")

# range(start, stop, step)
# Count every 100 blocks — useful for sampling
for block in range(19_847_000, 19_848_000, 100):
    print(f"Checkpoint block: {block:,}")

0 1 2 3 4 
Processing block 19,847,290
Processing block 19,847,291
Processing block 19,847,292
Processing block 19,847,293
Processing block 19,847,294
Processing block 19,847,295
Checkpoint block: 19,847,000
Checkpoint block: 19,847,100
Checkpoint block: 19,847,200
Checkpoint block: 19,847,300
Checkpoint block: 19,847,400
Checkpoint block: 19,847,500
Checkpoint block: 19,847,600
Checkpoint block: 19,847,700
Checkpoint block: 19,847,800
Checkpoint block: 19,847,900


In [41]:
# Practical example: simulate checking multiple blocks
import random

print("Scanning blocks for large transactions...\n")
large_txns_found = []

for block_num in range(19_847_000, 19_847_010):
    # Simulate finding transactions in this block
    # (In real code, you'd call web3.eth.get_block(block_num) here)
    simulated_value_eth = random.uniform(0, 50)

    if simulated_value_eth > 20:
        large_txns_found.append({
            "block": block_num,
            "value_eth": simulated_value_eth
        })
        print(f"  🚨 Block {block_num:,} — Large tx: {simulated_value_eth:.2f} ETH")
    else:
        print(f"  ✅ Block {block_num:,} — Normal activity")

print(f"\nFound {len(large_txns_found)} large transactions in scan range.")

Scanning blocks for large transactions...

  🚨 Block 19,847,000 — Large tx: 48.35 ETH
  ✅ Block 19,847,001 — Normal activity
  🚨 Block 19,847,002 — Large tx: 25.84 ETH
  🚨 Block 19,847,003 — Large tx: 36.68 ETH
  🚨 Block 19,847,004 — Large tx: 43.64 ETH
  🚨 Block 19,847,005 — Large tx: 28.72 ETH
  ✅ Block 19,847,006 — Normal activity
  🚨 Block 19,847,007 — Large tx: 21.08 ETH
  🚨 Block 19,847,008 — Large tx: 44.78 ETH
  🚨 Block 19,847,009 — Large tx: 21.25 ETH

Found 8 large transactions in scan range.


## 5. break and continue

Two special keywords that change how a loop runs:

- **`break`** — stop the loop immediately, exit it entirely
- **`continue`** — skip the rest of this iteration, jump to the next one

**Blockchain use case for `break`:** Stop scanning once you find what you're looking for.
**Blockchain use case for `continue`:** Skip failed or irrelevant transactions in a batch.

In [45]:
# break — stop as soon as we find a whale transaction
transactions = [
    {"hash": "0xaaa", "value_eth": 0.1,   "status": "success"},
    {"hash": "0xbbb", "value_eth": 0.5,   "status": "success"},
    {"hash": "0xccc", "value_eth": 250.0,  "status": "success"},  # whale!
    {"hash": "0xddd", "value_eth": 1.2,   "status": "success"},
    {"hash": "0xeee", "value_eth": 300,   "status": "success"},
]

WHALE_THRESHOLD = 100  # ETH

print("Scanning for first whale transaction...")
for tx in transactions:
    print(f"  Checking {tx['hash']}... {tx['value_eth']} ETH")
    if tx["value_eth"] >= WHALE_THRESHOLD:
        print(f"  🐋 Whale found! {tx['value_eth']} ETH — stopping scan.")
        break # no need to check the rest

Scanning for first whale transaction...
  Checking 0xaaa... 0.1 ETH
  Checking 0xbbb... 0.5 ETH
  Checking 0xccc... 250.0 ETH
  🐋 Whale found! 250.0 ETH — stopping scan.


In [46]:
# continue — skip failed transactions, only process successful ones
transactions = [
    {"hash": "0xaaa", "value_eth": 1.5,  "status": "success"},
    {"hash": "0xbbb", "value_eth": 0.3,  "status": "failed"},   # skip
    {"hash": "0xccc", "value_eth": 2.1,  "status": "success"},
    {"hash": "0xddd", "value_eth": 0.0,  "status": "failed"},   # skip
    {"hash": "0xeee", "value_eth": 5.0,  "status": "success"},
]

total_volume = 0
skipped = 0

for tx in transactions:
    if tx["status"] != "success":
        skipped += 1
        continue   # jump straight to next transaction

    # This only runs for successful transactions
    total_volume += tx["value_eth"]
    print(f"  ✅ {tx['hash']} | {tx['value_eth']} ETH")

print(f"\nTotal volume: {total_volume} ETH | Skipped: {skipped} failed txns")

  ✅ 0xaaa | 1.5 ETH
  ✅ 0xccc | 2.1 ETH
  ✅ 0xeee | 5.0 ETH

Total volume: 8.6 ETH | Skipped: 2 failed txns


## 6. while Loops

A `while` loop runs **as long as** a condition is True. Unlike `for` (which iterates a fixed sequence), `while` keeps going until something changes.

**Blockchain use case:** Polling for transaction confirmation. Retry logic for API calls.

In [ ]:
# while loop — basic structure
import time

block_number = 19_847_293
target_block = 19_847_300   # we want to wait until this block

print(f"Waiting for block {target_block:,}...")

# Simulating block progression
current_block = block_number

while current_block < target_block:
    current_block += 1   # simulate a new block arriving
    print(f"  Current block: {current_block:,}", end="")
    if current_block < target_block:
        print(" — waiting...")
    else:
        print(" — TARGET REACHED ✅")

print(f"Block {target_block:,} confirmed!")

In [ ]:
# while with break — retry loop for API calls
import random

MAX_RETRIES = 5
attempt = 0
success = False

print("Fetching ETH price from API...")

while attempt < MAX_RETRIES:
    attempt += 1
    print(f"  Attempt {attempt}/{MAX_RETRIES}...", end=" ")

    # Simulate API call that randomly fails
    api_succeeded = random.random() > 0.4   # 60% chance of success

    if api_succeeded:
        eth_price = 3247.85
        print(f"✅ Got price: ${eth_price:,.2f}")
        success = True
        break
    else:
        print("❌ API timeout")

if not success:
    print(f"Failed after {MAX_RETRIES} attempts — using cached price")

## 7. Nested Loops and Conditions

Loops inside loops, conditions inside loops — this is where it all comes together.

**Blockchain use case:** For each wallet, check each of their transactions.

In [ ]:
# Nested loop — multi-wallet, multi-token analysis
wallets = [
    {"address": "0xWallet1", "tokens": ["ETH", "USDC", "UNI"]},
    {"address": "0xWallet2", "tokens": ["ETH", "AAVE"]},
    {"address": "0xWallet3", "tokens": ["USDC", "DAI", "FRAX", "USDT"]},
]

STABLECOINS = ["USDC", "DAI", "FRAX", "USDT", "BUSD"]

print("Wallet token analysis:")
print("=" * 50)

for wallet in wallets:
    address = wallet["address"]
    tokens  = wallet["tokens"]

    stable_count = 0
    volatile_count = 0

    for token in tokens:
        if token in STABLECOINS:
            stable_count += 1
        else:
            volatile_count += 1

    # Risk profile based on holdings
    if stable_count == len(tokens):
        risk = "🟢 Stable only — very low risk"
    elif stable_count > volatile_count:
        risk = "🟡 Stable-heavy — low risk"
    elif volatile_count > stable_count:
        risk = "🟠 Volatile-heavy — medium risk"
    else:
        risk = "🔴 All volatile — high risk"

    print(f"{address} | {len(tokens)} tokens | {risk}")

## 8. Putting It Together — Wallet Activity Classifier

This week's mini-project. Given a list of transactions, analyse them and produce a full wallet report.

In [ ]:
# Full wallet analysis pipeline
transactions = [
    {"hash": "0xaaa", "value_eth": 0.5,    "status": "success", "type": "transfer",  "gas_gwei": 18},
    {"hash": "0xbbb", "value_eth": 15.0,   "status": "success", "type": "swap",      "gas_gwei": 22},
    {"hash": "0xccc", "value_eth": 0.001,  "status": "failed",  "type": "transfer",  "gas_gwei": 15},
    {"hash": "0xddd", "value_eth": 250.0,  "status": "success", "type": "transfer",  "gas_gwei": 45},
    {"hash": "0xeee", "value_eth": 0.05,   "status": "success", "type": "swap",      "gas_gwei": 20},
    {"hash": "0xfff", "value_eth": 1.2,    "status": "success", "type": "liquidity", "gas_gwei": 30},
    {"hash": "0xggg", "value_eth": 0.0,    "status": "failed",  "type": "swap",      "gas_gwei": 12},
    {"hash": "0xhhh", "value_eth": 5.5,    "status": "success", "type": "swap",      "gas_gwei": 25},
]

wallet_address = "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045"

# --- Aggregate stats ---
total_txns    = len(transactions)
success_txns  = 0
failed_txns   = 0
total_volume  = 0
largest_tx    = 0
type_counts   = {}
gas_prices    = []

for tx in transactions:
    # Count by status
    if tx["status"] == "success":
        success_txns += 1
        total_volume += tx["value_eth"]
        if tx["value_eth"] > largest_tx:
            largest_tx = tx["value_eth"]
    else:
        failed_txns += 1
        continue   # don't count failed txns in type breakdown

    # Count by type
    tx_type = tx["type"]
    if tx_type in type_counts:
        type_counts[tx_type] += 1
    else:
        type_counts[tx_type] = 1

    gas_prices.append(tx["gas_gwei"])

# --- Derived metrics ---
success_rate    = (success_txns / total_txns) * 100
avg_gas         = sum(gas_prices) / len(gas_prices) if gas_prices else 0
avg_tx_size     = total_volume / success_txns if success_txns > 0 else 0

# --- Wallet classification ---
if largest_tx >= 100:
    wallet_type = "🐋 Whale"
elif "swap" in type_counts and type_counts.get("swap", 0) >= 3:
    wallet_type = "⚡ DeFi Trader"
elif "liquidity" in type_counts:
    wallet_type = "💧 Liquidity Provider"
else:
    wallet_type = "👤 Regular User"

# --- Print report ---
print("=" * 52)
print(f"  WALLET REPORT")
print(f"  {wallet_address[:18]}...{wallet_address[-6:]}")
print("=" * 52)
print(f"  Classification : {wallet_type}")
print(f"  Total Txns     : {total_txns}")
print(f"  Success Rate   : {success_rate:.1f}%  ({success_txns} ok / {failed_txns} failed)")
print(f"  Total Volume   : {total_volume:.4f} ETH")
print(f"  Largest Tx     : {largest_tx:.4f} ETH")
print(f"  Avg Tx Size    : {avg_tx_size:.4f} ETH")
print(f"  Avg Gas Price  : {avg_gas:.1f} Gwei")
print(f"  Activity Mix   :", end="")
for tx_type, count in type_counts.items():
    print(f"  {count}x {tx_type}", end="")
print()
print("=" * 52)

## Summary

| Concept | Syntax | Blockchain use |
|---------|--------|----------------|
| `if/elif/else` | `if price > 5000:` | Token labeling, risk tiers |
| Comparison | `==`, `!=`, `>`, `<`, `>=`, `<=` | Threshold checks |
| Logical | `and`, `or`, `not`, `in` | Multi-condition filtering |
| `for` loop | `for tx in transactions:` | Batch processing |
| `enumerate()` | `for i, tx in enumerate(list):` | Indexed iteration |
| `range()` | `range(start, stop, step)` | Block range scanning |
| `break` | stop loop early | Find first match |
| `continue` | skip current item | Filter failed txns |
| `while` | `while block < target:` | Polling, retries |
| Nesting | loop inside loop | Multi-wallet analysis |

**SQL parallel recap:**
- `if/elif/else` ≈ `CASE WHEN ... THEN ... ELSE ... END`
- `for tx in transactions` ≈ iterating over result rows
- `continue` on failed txns ≈ `WHERE status = 'success'`
- Accumulating `total_volume` in a loop ≈ `SUM(value_eth)`

---

## What's next

**Week 4 — Data Structures:** Lists, tuples, dictionaries, and sets in depth. You'll build an on-chain portfolio tracker using nothing but Python's built-in data structures.

---

**Your task before Week 4:**
1. Complete `exercises.py`
2. Extend the wallet classifier to also flag wallets with > 50% failed transactions as "⚠️ Suspicious"
3. Commit:
```bash
git add phase-1-python-fundamentals/week-03-control-structures/
git commit -m "phase-1/week-03: completed control structures lesson and exercises"
git push
```